In [ ]:
!pip install pingouin
import pandas as pd
import numpy as np
import re
from scipy.stats import ttest_ind
import pingouin as pg
pd.options.mode.chained_assignment = None

In [ ]:
df_raw = pd.read_excel('/content/anketa523184-2026-05-26.xlsx')

# информированное согласие
df_raw = df_raw[df_raw['Q1'] == 1]

# проверки на внимательность
df_raw = df_raw[(df_raw['Q11a'] == 3) & (df_raw['statementsd'] ==2) & (df_raw['discusse'] == 7)]

# дропауты
df_raw = df_raw[~df_raw.eq(-3).any(axis=1)]

# переименуем колонки

df_raw.rename(columns={'discussa': 'pol_talks', 'discussb': 'pol_talks_online',
                   'discussc': 'outgroup_contacts',
                   'discussd': 'outgroup_pol_talks',
                   'Q306': 'gender',
                   'Q307': 'age',
                   'Q308':'educ',
                   'Q309':'income',
                   'Q310': 'city'}, inplace=True)

In [ ]:
df_raw['age'].value_counts()

,count
age,
2,165
3,159
1,72
4,68
5,31
6,11


In [ ]:
df_raw['gender'].value_counts()

,count
gender,
1,268
2,238


In [ ]:
# тут чистим тех, кто не прошел проверки на внимательное чтение текстов
manip_check_map = {
    'Q17': 'G1A',   'Q42': 'G1G',   'Q46': 'G2A',   'Q71': 'G2G',
    'Q75': 'G3A',   'Q100': 'G3G',  'Q104': 'G4A',  'Q129': 'G4G',
    'Q133': 'G5A',  'Q158': 'G5G',  'Q162': 'G6A',  'Q187': 'G6G',
    'Q191': 'G7A',  'Q216': 'G7G',  'Q220': 'G8A',  'Q245': 'G8G',
    'Q249': 'G9A',  'Q274': 'G9G',  'Q278': 'G10A', 'Q303': 'G10G',
}

check_cols = list(manip_check_map.keys())
skipped, correct = -2, 2

shown = df_raw[check_cols] != skipped
failed = shown & (df_raw[check_cols] != correct)

df_raw['n_shown'] = shown.sum(axis=1)
df_raw['n_failed'] = failed.sum(axis=1)
df_raw['pass'] = df_raw['n_failed'] == 0

print(f'До: {len(df_raw)}')
print(f'Отсеяно: {(~df_raw["pass"]).sum()}')

df_raw = df_raw[df_raw['pass']].copy()
print(f'После: {len(df_raw)}')

До: 506
Отсеяно: 109
После: 397


In [ ]:
# удалим респондентов без позиции по вопросу поддержки власти
df_raw = df_raw[df_raw['supporta'] != 4]
print(f'Наблюдений: {df_raw.shape[0]}')

# создаем бинарную категорию: 0 -- не поддерживает руководство, 1 -- поддерживает
df_raw['support_bin'] = np.where(df_raw['supporta'].isin([1,2,3]), 0, 1)
print(df_raw['support_bin'].value_counts(normalize = True))

# очищенные данные
df = df_raw.reset_index(drop=True)
df['rid'] = df.index + 1   #id респондента

Наблюдений: 329
support_bin
1    0.668693
0    0.331307
Name: proportion, dtype: float64


In [ ]:
rename_map = {
    'Q306': 'gender',
    'Q307': 'age',
    'Q308': 'educ',
    'Q309': 'income',
    'Q310': 'city'
}

labels = {
    'gender': {1: 'Женский', 2: 'Мужской'},
    'age': {1: '18-24', 2: '25-34', 3: '35-44', 4: '45-54', 5: '55-64', 6: '65+'},
    'educ': {
        1: 'Начальное или ниже',
        2: 'Среднее (школа)',
        3: 'Среднее специальное (техникум)',
        4: 'Незаконченное высшее',
        5: 'Высшее'
    },
    'income': {
        1: 'Не хватает денег даже на еду',
        2: 'На продукты денег хватает, но покупка одежды уже затруднительна',
        3: 'Денег хватает на продукты и одежду, но покупка холодильника, телевизора, мебели для нас проблема',
        4: 'Мы можем без труда купить холодильник, телевизор, мебель, но на большее денег нет',
        5: 'Мы можем позволить себе практически все: машину, квартиру, дачу и многое другое'
    },
    'city': {
        1: 'Москва или Санкт‑Петербург',
        2: 'Город с населением свыше 1 миллиона человек (кроме Москвы и Санкт‑Петербурга)',
        3: 'Город с населением менее 1 млн человек',
        4: 'Посёлок городского типа',
        5: 'Деревня, село'
    }
}

question_text = {
    'gender': 'Пол',
    'age':    'Возраст',
    'educ':   'Образование',
    'income': 'Доходная группа',
    'city':   'Тип населённого пункта'
}

latex_blocks = []

for var in rename_map.values():
    col = pd.to_numeric(df[var], errors='coerce')
    col = col[col > 0].astype(int)
    n_total = len(col)

    counts = col.value_counts().sort_index()
    label_map = labels[var]

    rows = []
    for code, cnt in counts.items():
        label = label_map.get(code, f'Код {code}')
        pct = cnt / n_total * 100
        rows.append((label, cnt, pct))

    q_label = question_text[var]

    lines = []
    lines.append(r'\midrule')
    lines.append(rf'\multicolumn{{3}}{{l}}{{\textbf{{{q_label}}} \textit{{(N={n_total})}}}} \\')
    for label, cnt, pct in rows:
        lines.append(rf'  \quad {label} & {cnt} & {pct:.1f}\% \\')

    latex_blocks.append('\n'.join(lines))

print(r'\begin{table}[h]')
print(r'\centering')
print(r'\small')                          # чуть мельче шрифт
print(r'\setlength{\tabcolsep}{4pt}')     # уже колонки
print(r'\caption{Социально-демографические характеристики выборки}')
print(r'\label{tab:socdem}')
print(r'\begin{tabular}{lcc}')
print(r'\toprule')
print(r'\textbf{Категория} & \textbf{N} & \textbf{\%} \\')
print('\n'.join(latex_blocks))
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

\begin{table}[h]
\centering
\small
\setlength{\tabcolsep}{4pt}
\caption{Социально-демографические характеристики выборки}
\label{tab:socdem}
\begin{tabular}{lcc}
\toprule
\textbf{Категория} & \textbf{N} & \textbf{\%} \\
\midrule
\multicolumn{3}{l}{\textbf{Пол} \textit{(N=329)}} \\
  \quad Женский & 173 & 52.6\% \\
  \quad Мужской & 156 & 47.4\% \\
\midrule
\multicolumn{3}{l}{\textbf{Возраст} \textit{(N=329)}} \\
  \quad 18-24 & 47 & 14.3\% \\
  \quad 25-34 & 107 & 32.5\% \\
  \quad 35-44 & 103 & 31.3\% \\
  \quad 45-54 & 42 & 12.8\% \\
  \quad 55-64 & 21 & 6.4\% \\
  \quad 65+ & 9 & 2.7\% \\
\midrule
\multicolumn{3}{l}{\textbf{Образование} \textit{(N=329)}} \\
  \quad Начальное или ниже & 1 & 0.3\% \\
  \quad Среднее (школа) & 33 & 10.0\% \\
  \quad Среднее специальное (техникум) & 92 & 28.0\% \\
  \quad Незаконченное высшее & 40 & 12.2\% \\
  \quad Высшее & 163 & 49.5\% \\
\midrule
\multicolumn{3}{l}{\textbf{Доходная группа} \textit{(N=329)}} \\
  \quad Не хватает денег даже на еду & 

In [ ]:
df[['patra', 'imppatra']].corr()

,patra,imppatra
patra,1.000000,0.907869
imppatra,0.907869,1.000000


In [ ]:
df[['euthana', 'impeuthana']].corr()

,euthana,impeuthana
euthana,1.000000,0.766524
impeuthana,0.766524,1.000000


In [ ]:
df

,Q1,Q3a,bordera,impbordera,vpna,impvpna,patra,imppatra,Q11a,forcea,...,gender,age,educ,income,city,n_shown,n_failed,pass,support_bin,rid
0,1,5,4.0,5.0,2.0,3.0,6.0,6.0,3.0,5.0,...,1,2,3,3,3,2,0,True,1,1
1,1,5,1.0,2.0,1.0,5.0,1.0,3.0,3.0,1.0,...,2,3,2,2,3,2,0,True,0,2
2,1,5,4.0,4.0,1.0,2.0,2.0,2.0,3.0,5.0,...,1,3,3,3,4,2,0,True,1,3
3,1,5,4.0,4.0,1.0,1.0,4.0,2.0,3.0,2.0,...,2,4,5,3,3,2,0,True,0,4
4,1,5,4.0,5.0,1.0,2.0,5.0,6.0,3.0,6.0,...,2,3,3,2,1,2,0,True,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324,1,5,6.0,6.0,7.0,7.0,4.0,3.0,3.0,4.0,...,2,1,4,4,3,2,0,True,1,325
325,1,5,2.0,4.0,1.0,7.0,1.0,1.0,3.0,5.0,...,2,2,5,4,1,2,0,True,1,326
326,1,5,7.0,7.0,1.0,2.0,5.0,5.0,3.0,3.0,...,1,2,5,3,2,2,0,True,0,327
327,1,5,2.0,4.0,4.0,5.0,6.0,6.0,3.0,7.0,...,2,1,4,4,2,2,0,True,1,328


In [ ]:
a = df[df['support_bin'] == 0]['identb'].value_counts().sort_index()
print(f'Процент  тех, кто не поддерживает власть и относит себя к критикам: {a.loc[[5, 6, 7]].sum() / a.sum() * 100:.2f}')


b = df[df['support_bin'] == 1]['identa'].value_counts().sort_index()
print(f'Процент тех, кто поддерживает власть и относит себя к сторонникам: {b.loc[[5, 6, 7]].sum() / b.sum() * 100:.2f}')

Процент  тех, кто не поддерживает власть и относит себя к критикам: 56.88
Процент тех, кто поддерживает власть и относит себя к сторонникам: 67.73


In [ ]:
df['support_bin'].value_counts()

,count
support_bin,
1,220
0,109


In [ ]:
print(f"Альфа кронбаха для доверия: {pg.cronbach_alpha(df[[i for i in df.columns if i.startswith('trust')]].apply(pd.to_numeric, errors='coerce'))}")

df['trust'] = df[[i for i in df.columns if i.startswith('trust')]].mean(axis = 1)


print(f"Альфа кронбаха для федеральных сми: {pg.cronbach_alpha(df[[i for i in df.columns if i.startswith('tv')]].apply(pd.to_numeric, errors='coerce'))}")

df['fed_tv'] = df[[i for i in df.columns if i.startswith('tv')]].mean(axis = 1)

Альфа кронбаха для доверия: (np.float64(0.9688009824413015), array([0.963, 0.974]))
Альфа кронбаха для федеральных сми: (np.float64(0.925450975127211), array([0.91 , 0.938]))


In [ ]:
# теперь надо перенести в этот массив данные по кодам виньеток и перевести все в long формат
vignette = pd.read_excel('/content/виньетки_итог.xlsx')
vignette_data = vignette[['set_id', 'position_in_set', 'var_name', 'vignette_code', 'A_gender', 'A_age', 'B_gender', 'B_age']]

vignette_data['topic'] = vignette_data['vignette_code'].astype(str).apply(lambda code: 'nopos' if code.startswith('Nopos') else code[0])
vignette_data['combo'] = vignette_data['vignette_code'].astype(str).apply(lambda code: pd.NA if code.startswith('Nopos') else code[1:3])

In [ ]:
base_cols = ['rid', 'interesta', 'identa', 'identb', 'trust', 'pol_talks_online', 'outgroup_contacts', 'outgroup_pol_talks', 'pol_talks', 'fed_tv', 'support_bin',
             'patra', 'imppatra', 'euthana', 'impeuthana',  'gender', 'age', 'educ', 'income', 'city']
rows = []

for _, v in vignette_data.iterrows():
    for cand, suffix in [('A', 'a'), ('B', 'b')]:
        x = df[base_cols].copy()
        x['set_id'] = v['set_id']
        x['position_in_set'] = v['position_in_set']
        x['var_name'] = v['var_name']
        x['vignette_code'] = v['vignette_code']
        x['topic'] = v['topic']
        x['combo'] = v['combo']

        x['candidate'] = cand
        x["rating"] = df[f"{v['var_name']}{suffix}"]

        x['cand_support_bin'] = 1 if cand == 'A' else 0
        x['cand_gender'] = v[f'{cand}_gender']
        x['cand_age'] = v[f'{cand}_age']

        rows.append(x)

long = pd.concat(rows, ignore_index=True)

# удалим те виньетки, которые респу не показывались (rating != -2)
long = long[long['rating'] != -2]
# проверка
assert long["rating"].between(1, 11).all()

long['ingroup'] = (long['cand_support_bin'].eq(long['support_bin'])).astype(int)


long = (long.sort_values(['rid','vignette_code', 'candidate']).reset_index(drop=True))

In [ ]:
long

,rid,interesta,identa,identb,trust,pol_talks_online,outgroup_contacts,outgroup_pol_talks,pol_talks,fed_tv,...,var_name,vignette_code,topic,combo,candidate,rating,cand_support_bin,cand_gender,cand_age,ingroup
0,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,G3E,EafF30F68,E,af,A,6,1,женщина,30 лет,1
1,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,G3E,EafF30F68,E,af,B,4,0,женщина,68 лет,0
2,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,G3G,EfaF30M46,E,fa,A,6,1,женщина,30 лет,1
3,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,G3G,EfaF30M46,E,fa,B,6,0,мужчина,46 лет,0
4,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,G3A,EffM30M46,E,ff,A,6,1,мужчина,30 лет,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4601,329,7.0,1,7,1.666667,1,4,4,4,2.333333,...,G5G,NoposM68M30,nopos,NaN,B,11,0,мужчина,30 лет,1
4602,329,7.0,1,7,1.666667,1,4,4,4,2.333333,...,G5B,PaaF46F68,P,aa,A,5,1,женщина,46 лет,0
4603,329,7.0,1,7,1.666667,1,4,4,4,2.333333,...,G5B,PaaF46F68,P,aa,B,11,0,женщина,68 лет,1
4604,329,7.0,1,7,1.666667,1,4,4,4,2.333333,...,G5F,PfaF68M30,P,fa,A,1,1,женщина,68 лет,0


In [ ]:
long['ingroup'].value_counts()

,count
ingroup,
1,2303
0,2303


In [ ]:
# добавляем бинарные позиции респондентов в массив
long['resp_issue_pos'] = pd.NA

long.loc[long['topic'].eq('P') & long['patra'].lt(4), 'resp_issue_pos'] = 0
long.loc[long['topic'].eq('P') & long['patra'].gt(4), 'resp_issue_pos'] = 1

long.loc[long['topic'].eq('E') & long['euthana'].lt(4), 'resp_issue_pos'] = 0
long.loc[long['topic'].eq('E') & long['euthana'].gt(4), 'resp_issue_pos'] = 1

In [ ]:
# добавляем бинарные позиции кандидатов по теме
long['cand_issue_pos'] = pd.NA

m = long['topic'].ne('nopos')
long.loc[m & long['candidate'].eq('A'), 'cand_issue_pos'] = (long.loc[m, 'combo'].str[0].eq('f').astype(int))
long.loc[m & long['candidate'].eq('B'), 'cand_issue_pos'] = (long.loc[m, 'combo'].str[1].eq('f').astype(int))

assert long.loc[m, 'combo'].str.len().eq(2).all(), "Неожиданный формат combo"
assert long.loc[m, 'combo'].str.match(r'^[fa]{2}$').all(), "Неожиданные символы в combo"

In [ ]:
# добавляем индикаторы совпадения позиций респондента и кандидата
long["issue_match"] = pd.NA
m = long["topic"].ne("nopos") & long["resp_issue_pos"].notna()
long.loc[m, "issue_match"] = (long.loc[m, "resp_issue_pos"].astype(int) == long.loc[m, "cand_issue_pos"].astype(int)).astype(int)

На этом этапе мы извлекли все необходимые нам переменные. Сейчас в массиве есть следующие переменные:

1. `support_bin` - поддержка респондентом руководства страны (0 - не поддерживает, 1 - поддерживает)

2. `patra` и `euthana` - позиции респондента по темам патриотического воспитания и эвтаназии соответственно в 7-бальной шкале

3. `imppatra` и `impeuthana` - важность для респондента тем патриотического воспитания и эвтаназии соответственно в 7-бальной шкале

4. `Q306 - Q310` - соц дем характеристики

5. `var_name` и `vignette_code` - технические переменные, из которых извлекалась информация: var_name - индикатор группы, куда попал респондент, vignette_code - какая респонденту досталась виньетка (тут указана тема, позиции, пол и возраст кандидатов)

6. `topic` - тема, которой посвящена виньетка: P - патриотическое воспитание, E - эвтаназия, nopos - нет темы

7. `combo` - комбинация позиций кандидатов в виньетке, тоже техническая деталь

8. `candidate` - кандидат A или B

9. `rating` - **зависимая переменная** оценка кандидата

10. `cand_support_bin` - поддержка кандидатом руководства страны (A всегда 1 - поддерживает, B всегда 0 - не поддерживает)

11. `cand_gender`	и `cand_age`- пол и возраст кандидата

12. `ingroup` - индикатор того, принадлежит ли кандидат к ин-группе респондента

13. `resp_issue_pos` - бинарная позиция респондента по теме виньетки (0 - не поддерживает, 1 - поддерживает и  nan, если выбрал вариант Не уверен(а) или если виньетка без темы (контрольная группа))

14. `cand_issue_pos` -  бинарная позиция кандидата по теме виньетки (0 - не поддерживает (в vignette_code обозначен как a), 1 - поддерживает (f) и  nan, если виньетка без темы (контрольная группа))

15. `issue_match` - совпадает ли респонеднт по своей позиции с кандидатом

In [ ]:
# теперь добавим pair_level переменные: rating_ingroup, rating_outgroup, ap, ingroup_mismatch, outgroup_match, а также хар-ки кандидатов

keys = ["rid", "var_name"]
# сейчас в каждой виньетке должно быть ровно 2 строки
assert long.groupby(keys).size().eq(2).all()

# один кандидат ин-группы и один кандидат аут-группы
assert long.groupby(keys)["ingroup"].sum().eq(1).all()

# и оценки должны быть от 1 до 11
assert long["rating"].between(1, 11).all()

In [ ]:
# возраст и пол на уровне пары

keys = ["rid", "var_name"]
long["cand_age"] = pd.to_numeric(long["cand_age"].astype(str).str.split().str[0], errors="coerce")

assert long.groupby(keys).size().eq(2).all()
assert long.groupby(keys)["ingroup"].sum().eq(1).all()


age_pair = (long.pivot(index=keys, columns="ingroup", values="cand_age").rename(columns={0: "age_outgroup",1: "age_ingroup"}).reset_index())
age_pair["age_gap"] = (age_pair["age_ingroup"] - age_pair["age_outgroup"])



gender_pair = (long.pivot(index=keys, columns="ingroup", values="cand_gender").rename(columns={0: "gender_outgroup",1: "gender_ingroup"}).reset_index())
gender_pair["same_gender_pair"] = (gender_pair["gender_ingroup"].eq(gender_pair["gender_outgroup"])).astype(int)


candidate_pair_vars = age_pair.merge(gender_pair, on=keys, how="left")

long = long.merge(candidate_pair_vars,on=keys,how="left")


assert long["age_ingroup"].notna().all()
assert long["age_outgroup"].notna().all()
assert long["gender_ingroup"].notna().all()
assert long["gender_outgroup"].notna().all()


long[[
        "rid",
        "var_name",
        "candidate",
        "ingroup",
        "cand_age",
        "cand_gender",
        "age_ingroup",
        "age_outgroup",
        "age_gap",
        "gender_ingroup",
        "gender_outgroup",
        "same_gender_pair"]].head(12)

,rid,var_name,candidate,ingroup,cand_age,cand_gender,age_ingroup,age_outgroup,age_gap,gender_ingroup,gender_outgroup,same_gender_pair
0,1,G3E,A,1,30,женщина,30,68,-38,женщина,женщина,1
1,1,G3E,B,0,68,женщина,30,68,-38,женщина,женщина,1
2,1,G3G,A,1,30,женщина,30,46,-16,женщина,мужчина,0
3,1,G3G,B,0,46,мужчина,30,46,-16,женщина,мужчина,0
4,1,G3A,A,1,30,мужчина,30,46,-16,мужчина,мужчина,1
5,1,G3A,B,0,46,мужчина,30,46,-16,мужчина,мужчина,1
6,1,G3F,A,1,46,мужчина,46,68,-22,мужчина,мужчина,1
7,1,G3F,B,0,68,мужчина,46,68,-22,мужчина,мужчина,1
8,1,G3C,A,1,30,мужчина,30,30,0,мужчина,женщина,0
9,1,G3C,B,0,30,женщина,30,30,0,мужчина,женщина,0


In [ ]:
ratings_pair = (long.pivot(index=keys, columns="ingroup", values="rating").rename(columns={0: "rating_outgroup", 1: "rating_ingroup"}).reset_index())
ratings_pair["ap"] = (ratings_pair["rating_ingroup"] - ratings_pair["rating_outgroup"])

ratings_pair

ingroup,rid,var_name,rating_outgroup,rating_ingroup,ap
0,1,G3A,7,6,-1
1,1,G3B,5,7,2
2,1,G3C,11,3,-8
3,1,G3D,4,8,4
4,1,G3E,4,6,2
...,...,...,...,...,...
2298,329,G5C,1,11,10
2299,329,G5D,1,11,10
2300,329,G5E,1,11,10
2301,329,G5F,1,11,10


In [ ]:
matches_pair = (long.pivot(index=keys, columns="ingroup", values="issue_match").rename(columns={0: "issue_match_outgroup",1: "issue_match_ingroup"}).reset_index())
pair_vars = ratings_pair.merge(matches_pair, on=keys, how="left")

In [ ]:
long = long.merge(pair_vars[keys + [
            "rating_ingroup",
            "rating_outgroup",
            "ap",
            "issue_match_ingroup",
            "issue_match_outgroup"
        ]],on=keys,how="left")

In [ ]:
# добавляем важность темы
long['resp_issue_importance'] = pd.NA
long.loc[long['topic'].eq('P'), 'resp_issue_importance'] = long.loc[long['topic'].eq('P'), 'imppatra']
long.loc[long['topic'].eq('E'), 'resp_issue_importance'] = long.loc[long['topic'].eq('E'), 'impeuthana']

In [ ]:
long.columns

Index(['rid', 'interesta', 'identa', 'identb', 'trust', 'pol_talks_online',
       'outgroup_contacts', 'outgroup_pol_talks', 'pol_talks', 'fed_tv',
       'support_bin', 'patra', 'imppatra', 'euthana', 'impeuthana', 'gender',
       'age', 'educ', 'income', 'city', 'set_id', 'position_in_set',
       'var_name', 'vignette_code', 'topic', 'combo', 'candidate', 'rating',
       'cand_support_bin', 'cand_gender', 'cand_age', 'ingroup',
       'resp_issue_pos', 'cand_issue_pos', 'issue_match', 'age_outgroup',
       'age_ingroup', 'age_gap', 'gender_outgroup', 'gender_ingroup',
       'same_gender_pair', 'rating_ingroup', 'rating_outgroup', 'ap',
       'issue_match_ingroup', 'issue_match_outgroup', 'resp_issue_importance'],
      dtype='object')

In [ ]:
# проверки
# ap должна быть у всех строк, потому что оценки есть у всех виньеток
assert long["ap"].notna().all()

# Внутри одной виньетки ap должна быть одинаковой в двух строках:
# строке кандидата A и строке кандидата B
assert long.groupby(keys)["ap"].nunique().eq(1).all()

# В nopos-виньетках issue_match отсутствует,
# значит ingroup_mismatch и outgroup_match тоже должны быть NA
assert long.loc[long["topic"].eq("nopos"), "issue_match_ingroup"].isna().all()
assert long.loc[long["topic"].eq("nopos"), "issue_match_outgroup"].isna().all()

In [ ]:
long.shape

(4606, 47)

In [ ]:
long.to_excel('long_ИТОГ.xlsx')

Переменные, которые появились после этого шага:

1. `age_outgroup` - возраст кандидата из аут-группы
2. `age_ingroup` - возраст кандидата из ин-группы
3. `age_gap`- разница возрастов
4. `gender_outgroup` - пол кандидата из аут-группы
5. `gender_ingroup`- пол кандидата из ин-группы
6. `same_gender_pair`- 0 - кандидаты разного пола, 1 - совпадают

------------------
**Зависимая переменная**

7. `rating_ingroup` - оценка кандидата из ин-группы
8. `rating_outgroup` - оценка кандидата из аут-группы
8. `ap` - аффективная поляризация

-------------------------------

9. `issue_match_ingroup` - совпадение позиции с кандидатом из ин группы
10. `issue_match_outgroup`  - совпадение позиции с кандидатом из аут группы
12. `resp_issue_importance` - важность темы виньетки для респондента

In [ ]:
# теперь сделаем датафрейм pair, где структура будет одна строка - одна виньетка. все что нужно для этого дф, подготовили выше
keys = ["rid", "var_name"]

# в каждой паре должно быть ровно 2 строки: кандидат A и кандидат B
assert long.groupby(keys).size().eq(2).all()

# в каждой паре должен быть один кандидат ин-группы и один кандидат аут-группы
assert long.groupby(keys)["ingroup"].sum().eq(1).all()

# ap должна быть одинаковой в двух строках одной пары
assert long.groupby(keys)["ap"].nunique(dropna=False).eq(1).all()


# удаляем переменные уровня отдельного кандидата

candidate_level_cols = [
    "candidate",
    "rating",
    "cand_support_bin",
    "cand_gender",
    "cand_age",
    "cand_issue_pos",
    "issue_match",
    "ingroup",
]

pair_base = long.drop(columns=[c for c in candidate_level_cols if c in long.columns],errors="ignore")


# проверяем, что внутри пары больше ничего не различается
nunique_by_pair = pair_base.groupby(keys).nunique(dropna=False)

problem_cols = nunique_by_pair.columns[nunique_by_pair.gt(1).any()].tolist()
assert len(problem_cols) == 0


# оставляем одну строку на rid × var_name
pair = (pair_base.drop_duplicates(subset=keys).reset_index(drop=True))



assert pair.groupby(keys).size().eq(1).all()
assert pair.shape[0] * 2 == long.shape[0]
assert pair["ap"].notna().all()

print(f"long shape: {long.shape}")
print(f"pair shape: {pair.shape}")

pair.head()

long shape: (4606, 47)
pair shape: (2303, 39)


,rid,interesta,identa,identb,trust,pol_talks_online,outgroup_contacts,outgroup_pol_talks,pol_talks,fed_tv,...,age_gap,gender_outgroup,gender_ingroup,same_gender_pair,rating_ingroup,rating_outgroup,ap,issue_match_ingroup,issue_match_outgroup,resp_issue_importance
0,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,-38,женщина,женщина,1,6,4,2,<NA>,<NA>,4.0
1,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,-16,мужчина,женщина,0,6,6,0,<NA>,<NA>,4.0
2,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,-16,мужчина,мужчина,1,6,7,-1,<NA>,<NA>,4.0
3,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,-22,мужчина,мужчина,1,6,6,0,<NA>,<NA>,<NA>
4,1,4.0,5,2,5.5,1,3,3,3,2.333333,...,0,женщина,мужчина,0,3,11,-8,0,1,6.0


In [ ]:
pair.to_excel('pair_ИТОГ.xlsx')